# Synthèse des résultats

---
## 1. Imports et chargement des données

In [1]:
import joblib
import warnings
import numpy as np
import pandas as pd

from html import escape
from pathlib import Path
from sklearn.pipeline import Pipeline
from IPython.display import display, HTML
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, average_precision_score,
)
import xgboost


warnings.filterwarnings("ignore", category=UserWarning)

MODE_AFFICHAGE = "html"

DATA_DIR   = Path("../data")
MODELS_DIR = Path("../models")

X_train = pd.read_csv(DATA_DIR / "weatherAUS_X_train.csv")
X_test  = pd.read_csv(DATA_DIR / "weatherAUS_X_test.csv")
y_train = pd.read_csv(DATA_DIR / "weatherAUS_y_train.csv").squeeze("columns").astype(int)
y_test  = pd.read_csv(DATA_DIR / "weatherAUS_y_test.csv").squeeze("columns").astype(int)

RESULTATS = {}

# Bloc de features d'ingénierie, rempli par la section 1.1. Laissé à None, les
# modèles qui n'attendent que la matrice du pipeline s'évaluent quand même.
EXTRA_TRAIN = EXTRA_TEST = None

print(f"X_train : {X_train.shape[0]:,} lignes x {X_train.shape[1]} colonnes")
print(f"X_test  : {X_test.shape[0]:,} lignes x {X_test.shape[1]} colonnes")
print(f"Taux de pluie train : {y_train.mean():.2%} | test : {y_test.mean():.2%}")

X_train : 113,506 lignes x 29 colonnes
X_test  : 25,314 lignes x 29 colonnes
Taux de pluie train : 21.96% | test : 22.69%


### 1.1. Reconstruction du bloc d'ingénierie XGBoost

Les bundles `EXP-007` attendent **44 variables** ; `weatherAUS_X_test.csv` n'en porte que 23.
Les 21 manquantes sont les blocs d'ingénierie construits par la série EXP-002 → 007, et elles ne
sont pas reconstructibles depuis `X_train` / `X_test` : ces matrices sont déjà passées au
`RobustScaler` et ne portent ni `Date` ni `Location`.

| bloc | n | source | contenu |
|---|---|---|---|
| intra-journalier | 2 | `weatherAUS_MICE_cyclic.csv` (**non scalé**) | `Pressure_diff`, `Temp_diff` |
| temporel | 12 | `weatherAUS_trend_grouped.csv` (**pré-imputation**) | 7 `*_lag1`, `Pressure_3d_chg`, `Humidity_3d_mean`, `Rainfall_3d_sum`, `RainToday_lag1`, `days_since_rain` |
| spatial | 7 | idem, graphe `K3_west` | 6 agrégats `nbr_*` des 3 voisines à l'ouest, `nearest_dist` |

La cellule ci-dessous reprend à l'identique les cellules 6 et 10 de
`XGBoost_Sath/EXP-007_XGBoost_spatial_tuned.ipynb`. Deux points sont critiques :

- toute feature d'ingénierie est calculée sur des valeurs **brutes** — le `RobustScaler` divise
  chaque colonne par son propre IQR, donc une différence de colonnes déjà scalées n'est pas la
  grandeur attendue par le modèle ;
- les lags et voisins viennent du frame **avant** imputation, pour qu'aucune valeur reconstruite
  par MICE ne remonte dans une variable retardée.

Le résultat est vérifié : réévalué ainsi, `EXP-007_02_tuned__chronological` retrouve exactement
les métriques enregistrées dans son bundle (F1 0.6843 / ROC 0.8964 / PR 0.7596).

Les NaN résiduels (premiers jours de chaque station, manquants d'origine) sont laissés tels quels :
XGBoost les gère nativement et EXP-003 a montré que les imputer n'apporte rien.

In [2]:
TEST_CUTOFF    = pd.Timestamp("2016-01-01")
GRAPHE_SPATIAL = "K3_west"   # graphe retenu par EXP-007 (clé 'spatial_graph' des bundles)

# --- Index (Date, Location) aligné ligne à ligne sur X_train / X_test ---------
# weatherAUS_MICE_cyclic.csv est trié par Date et le pipeline y applique le même
# TEST_CUTOFF : ses len(X_train) premières lignes SONT celles de X_train.
meta = pd.read_csv(DATA_DIR / "weatherAUS_MICE_cyclic.csv",
                   usecols=["Date", "Location", "Pressure9am", "Pressure3pm", "Temp9am", "Temp3pm"],
                   parse_dates=["Date"])
meta = meta[["Date", "Location", "Pressure9am", "Pressure3pm", "Temp9am", "Temp3pm"]]

N_TRAIN = len(X_train)
assert len(meta) == N_TRAIN + len(X_test), "X_train/X_test ne recouvrent pas MICE_cyclic"
assert (meta["Date"].iloc[:N_TRAIN] < TEST_CUTOFF).all(), "lignes de train postérieures au seuil"
assert (meta["Date"].iloc[N_TRAIN:] >= TEST_CUTOFF).all(), "lignes de test antérieures au seuil"

# Frame AVANT imputation (mêmes lignes, NaN d'origine conservés) : aucune valeur
# reconstruite par MICE ne peut ainsi remonter dans un lag ou une moyenne voisine.
pre = pd.read_csv(DATA_DIR / "weatherAUS_trend_grouped.csv", parse_dates=["Date"])
assert len(pre) == len(meta), "weatherAUS_trend_grouped.csv désaligné du pipeline"


def align_to_pipeline(frame, cols):
    """Réaligne un bloc indexé par (Date, Location) sur l'ordre des lignes du pipeline."""
    key = pd.MultiIndex.from_frame(frame[["Date", "Location"]])
    assert key.is_unique, "clé (Date, Location) non unique dans le bloc de features"
    out = frame.set_axis(key)[cols].reindex(pd.MultiIndex.from_frame(meta[["Date", "Location"]]))
    return out.reset_index(drop=True)


EXTRA = pd.DataFrame(index=range(len(meta)))

# --- 2 features de tendance intra-journalière (valeurs NON scalées) ----------
EXTRA["Pressure_diff"] = (meta["Pressure3pm"] - meta["Pressure9am"]).to_numpy()
EXTRA["Temp_diff"]     = (meta["Temp3pm"]     - meta["Temp9am"]).to_numpy()

# --- 12 features temporelles (uniquement des jours <= t-1, par Location) -----
pre_s = pre.sort_values(["Location", "Date"]).copy()
g     = pre_s.groupby("Location", sort=False)

LAG_COLS = ["Humidity3pm", "Pressure3pm", "Rainfall", "Cloud3pm", "Sunshine", "Temp3pm", "WindGustSpeed"]
feat = pd.DataFrame({"Date": pre_s["Date"].values, "Location": pre_s["Location"].values})
for c in LAG_COLS:
    feat[f"{c}_lag1"] = g[c].shift(1).values

feat["Pressure_3d_chg"]  = (g["Pressure3pm"].shift(1) - g["Pressure3pm"].shift(4)).values
feat["Humidity_3d_mean"] = g["Humidity3pm"].apply(lambda s: s.shift(1).rolling(3).mean()).values
feat["Rainfall_3d_sum"]  = g["Rainfall"].apply(lambda s: s.shift(1).rolling(3).sum()).values

rt     = pre_s["RainToday"].astype(int)          # booléen dans le frame du pipeline
rt_lag = rt.groupby(pre_s["Location"], sort=False).shift(1)
feat["RainToday_lag1"] = rt_lag.values


def _days_since(s):
    out = np.empty(len(s)); cnt = 9999.0
    for i, v in enumerate(s.to_numpy()):
        out[i] = cnt
        cnt = 0.0 if v == 1 else (cnt + 1 if cnt < 9999 else 9999)
    return out


feat["days_since_rain"] = (rt_lag.fillna(0).groupby(pre_s["Location"], sort=False)
                                 .transform(lambda s: _days_since(s)).values)

TEMPORAL_FEATURES = [c for c in feat.columns if c not in ("Date", "Location")]
EXTRA[TEMPORAL_FEATURES] = align_to_pipeline(feat, TEMPORAL_FEATURES)

# --- 7 features spatiales : agrégats des 3 stations voisines à l'ouest -------
COORDS = {
    "Adelaide": (-34.9285, 138.6007), "Albany": (-35.0269, 117.8837), "Albury": (-36.0737, 146.9135),
    "AliceSprings": (-23.6980, 133.8807), "BadgerysCreek": (-33.8829, 150.7609), "Ballarat": (-37.5622, 143.8503),
    "Bendigo": (-36.7570, 144.2794), "Brisbane": (-27.4698, 153.0251), "Cairns": (-16.9186, 145.7781),
    "Canberra": (-35.2809, 149.1300), "Cobar": (-31.4958, 145.8389), "CoffsHarbour": (-30.2986, 153.1094),
    "Dartmoor": (-37.9225, 141.2767), "Darwin": (-12.4634, 130.8456), "GoldCoast": (-28.0167, 153.4000),
    "Hobart": (-42.8821, 147.3272), "Katherine": (-14.4652, 132.2635), "Launceston": (-41.4332, 147.1441),
    "Melbourne": (-37.8136, 144.9631), "MelbourneAirport": (-37.6690, 144.8410), "Mildura": (-34.1855, 142.1625),
    "Moree": (-29.4639, 149.8407), "MountGambier": (-37.8284, 140.7820), "MountGinini": (-35.5294, 148.7723),
    "Newcastle": (-32.9283, 151.7817), "Nhil": (-36.3333, 141.6500), "NorahHead": (-33.2833, 151.5667),
    "NorfolkIsland": (-29.0408, 167.9547), "Nuriootpa": (-34.4700, 138.9960), "PearceRAAF": (-31.6676, 116.0292),
    "Penrith": (-33.7507, 150.6877), "Perth": (-31.9505, 115.8605), "PerthAirport": (-31.9385, 115.9672),
    "Portland": (-38.3609, 141.6041), "Richmond": (-33.6000, 150.7667), "Sale": (-38.1026, 147.0730),
    "SalmonGums": (-32.9815, 121.6438), "Sydney": (-33.8688, 151.2093), "SydneyAirport": (-33.9399, 151.1753),
    "Townsville": (-19.2590, 146.8169), "Tuggeranong": (-35.4244, 149.0888), "Uluru": (-25.3444, 131.0369),
    "WaggaWagga": (-35.1082, 147.3598), "Walpole": (-34.9777, 116.7338), "Watsonia": (-37.7100, 145.0830),
    "Williamtown": (-32.8150, 151.8428), "Witchcliffe": (-34.0261, 115.1003), "Wollongong": (-34.4278, 150.8931),
    "Woomera": (-31.1656, 136.8193),
}
assert set(meta["Location"].unique()) <= set(COORDS)
stations = sorted(COORDS)


def haversine(a, b):
    R = 6371.0; p1, p2 = np.radians(a[0]), np.radians(b[0])
    dphi = np.radians(b[0] - a[0]); dl = np.radians(b[1] - a[1])
    h = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(h))


def knn(K, west_only=False):
    nb, nd = {}, {}
    for s in stations:
        cand = sorted((haversine(COORDS[s], COORDS[t]), t) for t in stations if t != s)
        nd[s] = cand[0][0]
        if west_only:
            west = [t for _, t in cand if COORDS[t][1] < COORDS[s][1]]
            rest = [t for _, t in cand if COORDS[t][1] >= COORDS[s][1]]
            nb[s] = (west + rest)[:K]
        else:
            nb[s] = [t for _, t in cand][:K]
    return nb, nd


NEIGHBOURS   = knn(3, west_only=True)[0]   # GRAPHE_SPATIAL == 'K3_west'
NEAREST_DIST = knn(1)[1]

pre2       = pre.assign(_RainToday=pre["RainToday"].astype(int))
full_dates = pd.date_range(pre["Date"].min(), pre["Date"].max(), freq="D")
WIDE = {var: (pre2.pivot_table(index="Date", columns="Location", values=var, dropna=False)
                  .reindex(index=full_dates, columns=stations))
        for var in ["Rainfall", "Pressure3pm", "Humidity3pm", "_RainToday"]}
SPEC = [("Rainfall", False), ("Rainfall", True), ("Pressure3pm", False),
        ("Pressure3pm", True), ("Humidity3pm", False), ("_RainToday", False)]

bloc_spatial     = meta[["Date", "Location"]].copy()
SPATIAL_FEATURES = []
for var, lag in SPEC:
    wide = WIDE[var]
    agg  = pd.DataFrame({s: wide[NEIGHBOURS[s]].mean(axis=1) for s in stations}, index=wide.index)
    if lag:
        agg = agg.shift(1)
    agg.index.name = "Date"
    nom = f'nbr_{var.replace("_RainToday", "RainToday")}_{"t1" if lag else "t"}'
    bloc_spatial = bloc_spatial.merge(
        agg.reset_index().melt(id_vars="Date", var_name="Location", value_name=nom),
        on=["Date", "Location"], how="left")
    SPATIAL_FEATURES.append(nom)
bloc_spatial["nearest_dist"] = bloc_spatial["Location"].map(NEAREST_DIST)
SPATIAL_FEATURES.append("nearest_dist")
EXTRA[SPATIAL_FEATURES] = align_to_pipeline(bloc_spatial, SPATIAL_FEATURES)

# --- Découpage sur le même ordre chronologique que le pipeline ---------------
EXTRA_TRAIN = EXTRA.iloc[:N_TRAIN].reset_index(drop=True)
EXTRA_TEST  = EXTRA.iloc[N_TRAIN:].reset_index(drop=True)

print(f"Bloc d'ingénierie : {EXTRA.shape[1]} colonnes x {EXTRA.shape[0]:,} lignes "
      f"(train {len(EXTRA_TRAIN):,} / test {len(EXTRA_TEST):,})")
print(f"  intra-journalier (2) : Pressure_diff, Temp_diff")
print(f"  temporel        ({len(TEMPORAL_FEATURES):2d}) : {', '.join(TEMPORAL_FEATURES)}")
print(f"  spatial         ({len(SPATIAL_FEATURES):2d}) : {', '.join(SPATIAL_FEATURES)}  [graphe {GRAPHE_SPATIAL}]")
print(f"Part de NaN moyenne : {EXTRA.isna().mean().mean():.1%} (laissée à la gestion native de XGBoost)")

Bloc d'ingénierie : 21 colonnes x 138,820 lignes (train 113,506 / test 25,314)
  intra-journalier (2) : Pressure_diff, Temp_diff
  temporel        (12) : Humidity3pm_lag1, Pressure3pm_lag1, Rainfall_lag1, Cloud3pm_lag1, Sunshine_lag1, Temp3pm_lag1, WindGustSpeed_lag1, Pressure_3d_chg, Humidity_3d_mean, Rainfall_3d_sum, RainToday_lag1, days_since_rain
  spatial         ( 7) : nbr_Rainfall_t, nbr_Rainfall_t1, nbr_Pressure3pm_t, nbr_Pressure3pm_t1, nbr_Humidity3pm_t, nbr_RainToday_t, nearest_dist  [graphe K3_west]
Part de NaN moyenne : 6.0% (laissée à la gestion native de XGBoost)


---
## 2. Fonctions utilitaires

Les bundles n'ont pas tous la même forme selon le notebook qui les a produits :

| Origine | Contenu du fichier | Clé du seuil |
|---|---|---|
| `light_gbm`, `linear_svc` — baseline / grid search | modèle nu (pas de dictionnaire) | — |
| `light_gbm`, `linear_svc` — threshold | `{model, seuil, features}` | `seuil` |
| `logistic_regression` | `{model, threshold, features, scaler, metrics, ...}` | `threshold` |
| `xgboost` (`XGBoost_Sath/EXP-*`) | `{model, threshold, features, metrics, params, split, spatial_graph, ...}` | `threshold` |

Les fonctions ci-dessous normalisent ces variantes et gèrent les cas particuliers :
`Pipeline` (scaling interne), scaler externe embarqué, termes d'interaction à
reconstruire (`poly` / `top_k` / `cross_idx`), ensemble bootstrap (`model` est une
**liste** de modèles dont on moyenne les probabilités), et variables d'ingénierie
absentes de la matrice du pipeline (`extra`, cf. section 1.1).

In [3]:
def normaliser_bundle(objet):
    """Ramène un fichier sérialisé (modèle nu ou dictionnaire) à un dictionnaire homogène."""
    if isinstance(objet, dict) and "model" in objet:
        bundle = dict(objet)
    else:
        bundle = {"model": objet}
    # Uniformisation de la clé du seuil : 'seuil' (FR) et 'threshold' (EN) coexistent
    if "threshold" not in bundle and "seuil" in bundle:
        bundle["threshold"] = bundle["seuil"]
    return bundle


def _modele_reference(modele):
    """Renvoie un modèle représentatif (1er élément si c'est un ensemble bootstrap)."""
    return modele[0] if isinstance(modele, list) else modele


def preparer_X(bundle, X, extra=None):
    """Applique au jeu X exactement le prétraitement attendu par le modèle rechargé.

    `extra` est le bloc de features d'ingénierie construit en 1.1 : il n'est
    consulté que pour les variables absentes de X (cas des bundles XGBoost EXP-007).
    """
    modele = bundle["model"]
    ref    = _modele_reference(modele)
    notes  = []

    if isinstance(ref, Pipeline):
        Xp = X.copy()
        notes.append("Pipeline : scaling interne")
    else:
        scaler = bundle.get("scaler")
        if scaler is None and hasattr(ref, "coef_"):
            raise ValueError(
                "Modèle linéaire sans scaler dans le bundle — "
                "l'export attendu (voir 5_modelisation_linear_svc.ipynb) "
                "doit inclure la clé 'scaler'."
            )
        if scaler is not None:
            notes.append("scaler du bundle")
            Xp = pd.DataFrame(scaler.transform(X), columns=X.columns, index=X.index)
        else:
            Xp = X.copy()
            notes.append("features brutes")

    # Reconstruction des termes d'interaction si le bundle les décrit
    if "poly" in bundle and "top_k" in bundle:
        arr = bundle["poly"].transform(Xp[bundle["top_k"]])
        croix = pd.DataFrame(arr[:, bundle["cross_idx"]],
                             columns=bundle["cross_names"], index=Xp.index)
        Xp = pd.concat([Xp, croix], axis=1)
        notes.append("termes d'interaction reconstruits")

    # Réalignement sur la liste ordonnée de variables du bundle
    features = bundle.get("features")
    if features is not None and list(Xp.columns) != list(features):
        manquantes = [f for f in features if f not in Xp.columns]

        # Les bundles EXP-007 attendent 44 variables : les 21 absentes de la
        # matrice du pipeline viennent du bloc reconstruit en 1.1.
        if manquantes and extra is not None:
            recuperables = [f for f in manquantes if f in extra.columns]
            if recuperables:
                Xp = pd.concat([Xp, extra[recuperables].set_axis(Xp.index)], axis=1)
                notes.append(f"{len(recuperables)} variables d'ingénierie ajoutées")
                manquantes = [f for f in features if f not in Xp.columns]

        if manquantes:
            raise ValueError(f"{len(manquantes)} variable(s) attendue(s) introuvable(s) : {manquantes[:5]}")
        Xp = Xp[list(features)]
        notes.append("colonnes réalignées")

    return Xp, notes


def calculer_scores(modele, Xp):
    """Renvoie (scores de la classe positive, type de score)."""
    if isinstance(modele, list):                       # ensemble bootstrap
        return np.mean([m.predict_proba(Xp)[:, 1] for m in modele], axis=0), "proba"
    if hasattr(modele, "predict_proba"):
        return modele.predict_proba(Xp)[:, 1], "proba"
    if hasattr(modele, "decision_function"):
        return modele.decision_function(Xp), "decision"
    raise TypeError(f"{type(modele).__name__} n'expose ni predict_proba ni decision_function")


def resoudre_seuil(bundle, type_score):
    """Renvoie (seuil, origine, alerte éventuelle)."""
    defaut = 0.5 if type_score == "proba" else 0.0
    seuil  = bundle.get("threshold")
    if seuil is None:
        return defaut, "défaut", None
    alerte = None
    if type_score == "proba" and not (0.0 < seuil < 1.0):
        alerte = (f"seuil {seuil} hors de [0, 1] alors que le score est une PROBABILITÉ "
                  f"→ toutes les prédictions basculent dans une seule classe")
    return seuil, "bundle", alerte

### 2.1. Affichage et évaluation

Le format du tableau reprend celui utilisé dans les notebooks de modélisation.

In [4]:
SEP = "─" * 52

_STYLE = ("font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 12.5px; "
          "line-height: 1.35; white-space: pre; margin: 0 0 14px 0;")


def emettre(bloc):
    """Émet un bloc de texte comme sortie autonome (HTML) ou via stdout."""
    if MODE_AFFICHAGE == "html":
        display(HTML(f"<pre style='{_STYLE}'>{escape(bloc)}</pre>"))
    else:
        print(bloc + "\n")


def afficher_evaluation(nom_classe, variante, m, seuil, origine_seuil, type_score, notes, alerte):
    """Construit et émet le tableau de métriques Train / Test."""
    label_score = "probabilité" if type_score == "proba" else "decision_function"
    lignes = [
        SEP,
        f"{'ÉVALUATION DU MODÈLE — ' + nom_classe:^52}",
        f"{variante:^52}",
        SEP,
        f"{'Métrique':<28} {'Train':>10} {'Test':>10}",
        SEP,
        f"{'AUC-ROC':<28} {m['auc_train']:>10.4f} {m['auc_test']:>10.4f}",
        f"{'PR-AUC':<28} {m['pr_train']:>10.4f} {m['pr_test']:>10.4f}",
        SEP,
        f"{'F1-score (classe Yes)':<28} {m['f1_train']:>10.4f} {m['f1_test']:>10.4f}",
        SEP,
        f"{'Recall   (classe Yes)':<28} {m['recall_train']:>10.4f} {m['recall_test']:>10.4f}",
        f"{'Précision (classe Yes)':<28} {m['prec_train']:>10.4f} {m['prec_test']:>10.4f}",
        SEP,
        f"{'Seuil de décision':<28} {seuil:>10} ({origine_seuil}, {label_score})",
        f"{'Prétraitement':<28} {' + '.join(notes) if notes else 'aucun':>10}",
    ]
    if alerte:
        lignes += ["", f"  /!\\  ALERTE — {alerte}"]
    lignes.append(SEP)
    emettre("\n".join(lignes))


def classer_surapprentissage(ecart):
    """Classification de l'écart Train/Test sur l'AUC-ROC (convention du rapport)."""
    if ecart < 0.02:
        return "non"
    if ecart < 0.05:
        return "léger"
    if ecart < 0.10:
        return "modéré"
    return "fort"


def evaluer_fichier(chemin, dossier):
    """Recharge un fichier, réévalue le modèle, affiche le tableau et renvoie une ligne de synthèse."""
    bundle = normaliser_bundle(joblib.load(chemin))
    modele = bundle["model"]
    ref    = _modele_reference(modele)

    nom_classe = type(ref).__name__
    if isinstance(ref, Pipeline):
        nom_classe = type(ref.steps[-1][1]).__name__
    if isinstance(modele, list):
        nom_classe = f"Ensemble[{len(modele)} x {nom_classe}]"

    # EXTRA_TRAIN / EXTRA_TEST (section 1.1) fournissent les variables d'ingénierie
    # absentes de la matrice du pipeline ; ils valent None si 1.1 n'a pas été exécutée.
    Xtr, notes = preparer_X(bundle, X_train, EXTRA_TRAIN)
    Xte, _     = preparer_X(bundle, X_test,  EXTRA_TEST)

    s_train, type_score = calculer_scores(modele, Xtr)
    s_test,  _          = calculer_scores(modele, Xte)

    seuil, origine_seuil, alerte = resoudre_seuil(bundle, type_score)

    p_train = (s_train >= seuil).astype(int)
    p_test  = (s_test  >= seuil).astype(int)

    m = {
        "auc_train":    roc_auc_score(y_train, s_train),
        "auc_test":     roc_auc_score(y_test,  s_test),
        "pr_train":     average_precision_score(y_train, s_train),
        "pr_test":      average_precision_score(y_test,  s_test),
        "f1_train":     f1_score(y_train, p_train, pos_label=1, zero_division=0),
        "f1_test":      f1_score(y_test,  p_test,  pos_label=1, zero_division=0),
        "recall_train": recall_score(y_train, p_train, zero_division=0),
        "recall_test":  recall_score(y_test,  p_test,  zero_division=0),
        "prec_train":   precision_score(y_train, p_train, zero_division=0),
        "prec_test":    precision_score(y_test,  p_test,  zero_division=0),
    }

    variante = chemin.stem
    afficher_evaluation(nom_classe, variante, m, seuil, origine_seuil, type_score, notes, alerte)

    ecart = m["auc_train"] - m["auc_test"]
    return (dossier, variante), {
        "Famille": dossier, "Variante": variante, "Modèle": nom_classe,
        "Seuil": seuil, "Type score": type_score,
        "AUC-ROC Train": m["auc_train"], "AUC-ROC Test": m["auc_test"],
        "PR-AUC Test": m["pr_test"],
        "F1 Train": m["f1_train"], "F1 Test": m["f1_test"],
        "Recall Test": m["recall_test"], "Précision Test": m["prec_test"],
        "Écart AUC": ecart, "Surapprentissage": classer_surapprentissage(ecart),
        "Alerte": alerte or "",
    }


def evaluer_dossier(dossier):
    """Évalue tous les modèles d'un sous-dossier de ../models/ et met à jour RESULTATS."""
    chemin_dossier = MODELS_DIR / dossier
    if not chemin_dossier.is_dir():
        print(f"Dossier introuvable : {chemin_dossier}")
        return

    fichiers = sorted(list(chemin_dossier.glob("*.pkl")) + list(chemin_dossier.glob("*.joblib")))
    if not fichiers:
        print(f"Aucun modèle sérialisé dans {chemin_dossier}")
        return

    emettre("\n".join(["#" * 52,
                       f"#  {dossier.upper()}  —  {len(fichiers)} modèle(s)",
                       "#" * 52]))

    for fichier in fichiers:
        try:
            cle, ligne = evaluer_fichier(fichier, dossier)
            RESULTATS[cle] = ligne
        except Exception as e:
            emettre("\n".join([SEP,
                               f"  ÉCHEC — {fichier.name}",
                               f"  {type(e).__name__} : {e}",
                               SEP]))

---
## 3. Inventaire des modèles disponibles

In [5]:
if not MODELS_DIR.is_dir():
    print(f"Dossier introuvable : {MODELS_DIR.resolve()}")
else:
    sous_dossiers = sorted(d for d in MODELS_DIR.iterdir() if d.is_dir())
    print(f"{len(sous_dossiers)} famille(s) de modèles dans {MODELS_DIR.resolve()}\n")

    for d in sous_dossiers:
        fichiers = sorted(list(d.glob("*.pkl")) + list(d.glob("*.joblib")))
        lignes = [f"{d.name}/  ({len(fichiers)} fichier(s))"]
        for f in fichiers:
            taille = f.stat().st_size / 1024
            try:
                objet = joblib.load(f)
                if isinstance(objet, dict):
                    structure = "dict → " + ", ".join(sorted(objet.keys()))
                    contenu   = type(objet.get("model")).__name__
                else:
                    structure = "modèle nu"
                    contenu   = type(objet).__name__
                lignes.append(f"   {f.name:<44} {taille:>8.1f} Ko   [{contenu}]")
                lignes.append(f"   {'':<44} {'':>8}      {structure}")
            except Exception as e:
                lignes.append(f"   {f.name:<44} {taille:>8.1f} Ko   ILLISIBLE : {type(e).__name__}")
        emettre("\n".join(lignes))

2 famille(s) de modèles dans C:\dev\juin25_alt2_mle_meteo\models



---
## 4. Évaluation par famille de modèles

### 4.1. Régression logistique — `logistic_regression`

In [6]:
evaluer_dossier("logistic_regression")

### 4.2. SVM linéaire — `linear_svc`

In [7]:
evaluer_dossier("linear_svc")

Dossier introuvable : ..\models\linear_svc


### 4.3. Gradient boosting — `light_gbm`

In [8]:
evaluer_dossier("light_gbm")

Dossier introuvable : ..\models\light_gbm


### 4.4. Arbre de décision — `decision_tree`

In [9]:
evaluer_dossier("decision_tree")

Dossier introuvable : ..\models\decision_tree


### 4.5. Forêt aléatoire — `random_forest`

In [10]:
evaluer_dossier("random_forest")

Dossier introuvable : ..\models\random_forest


### 4.6. XGBoost — `xgboost`

Les bundles évalués ici viennent de `XGBoost_Sath/EXP-007_XGBoost_spatial_tuned.ipynb` et attendent
les 44 variables reconstruites en 1.1.

**Seuls les bundles `__chronological` ont leur place dans cette synthèse.** Les autres artefacts de
la série ne sont pas évaluables sur le découpage `X_train` / `X_test` de ce notebook :

- `__leave_stations_out` : entraîné sur 80 % des **stations**, toutes dates confondues — il a donc
  vu des lignes postérieures au `TEST_CUTOFF`, et son « AUC Test » serait gonflé ;
- `deploy_03_production_full_train` : entraîné sur la totalité du jeu — Train et Test seraient tous
  deux du train.

In [11]:
evaluer_dossier("xgboost")

### 4.7. CatBoost — `catboost`

In [12]:
evaluer_dossier("catboost")

Dossier introuvable : ..\models\catboost


---
## 5. Tableau récapitulatif

Synthèse de toutes les variantes évaluées, triées par AUC-ROC de test décroissante.
L'AUC-ROC gouverne la comparaison entre modèles, le PR-AUC vient la nuancer sous déséquilibre de classes, et le F1 sur la classe *Yes* mesure la performance au point de fonctionnement retenu.

In [13]:
if not RESULTATS:
    print("Aucun résultat — exécutez d'abord les cellules de la section 4.")
else:
    recap = pd.DataFrame(RESULTATS.values())

    # Grouper par famille : chaque famille est ordonnée par sa meilleure
    # AUC-ROC Test, puis les variantes sont triées à l'intérieur par
    # AUC-ROC Test, PR-AUC Test, F1 Test décroissants (recall/précision
    # ne sont pas des critères de décision, cf. 5.2 du rapport)
    ordre_familles = (recap.groupby("Famille")["AUC-ROC Test"].max()
                            .sort_values(ascending=False).index)
    recap["Famille"] = pd.Categorical(recap["Famille"], categories=ordre_familles, ordered=True)
    recap = (recap.sort_values(["Famille", "AUC-ROC Test", "PR-AUC Test", "F1 Test"],
                                ascending=[True, False, False, False])
                  .reset_index(drop=True))

    colonnes = ["Famille", "Variante", "Seuil", "AUC-ROC Train", "AUC-ROC Test",
                "PR-AUC Test", "F1 Train", "F1 Test", "Recall Test", "Précision Test",
                "Écart AUC", "Surapprentissage"]

    display(recap[["Famille", "Variante", "Seuil", "AUC-ROC Train", "AUC-ROC Test",
                   "PR-AUC Test", "F1 Train", "F1 Test", "Recall Test", "Précision Test",
                   "Écart AUC", "Surapprentissage"]]
            .style.format({c: "{:.4f}" for c in colonnes if c not in
                     ["Famille", "Variante", "Seuil", "Surapprentissage"]})
            .background_gradient(subset=["AUC-ROC Test", "PR-AUC Test", "F1 Test"], cmap="Greens"))

    alertes = recap[recap["Alerte"] != ""]
    if not alertes.empty:
        print(f"\n{len(alertes)} modèle(s) signalé(s) :")
        for _, r in alertes.iterrows():
            print(f"  - {r['Famille']}/{r['Variante']} : {r['Alerte']}")

,Famille,Variante,Seuil,AUC-ROC Train,AUC-ROC Test,PR-AUC Test,F1 Train,F1 Test,Recall Test,Précision Test,Écart AUC,Surapprentissage
0,xgboost,EXP-007_02_frozen__chronological,0.610000,0.9607,0.8966,0.7619,0.7948,0.6823,0.7124,0.6546,0.0641,modéré
1,xgboost,EXP-007_02_tuned__chronological,0.590000,0.9682,0.8964,0.7596,0.8156,0.6843,0.7171,0.6544,0.0719,modéré
2,logistic_regression,logistic_regression_interactions,0.670000,0.8822,0.8692,0.6990,0.6478,0.6404,0.6513,0.6298,0.0130,non
3,logistic_regression,logistic_regression_grid_search_tuned,0.670000,0.8776,0.8644,0.6900,0.6403,0.6355,0.6402,0.6308,0.0132,non
4,logistic_regression,logistic_regression_imbalance_balanced,0.500000,0.8776,0.8643,0.6900,0.6299,0.6191,0.7755,0.5152,0.0132,non
5,logistic_regression,logistic_regression_imbalance_smote,0.500000,0.8773,0.8643,0.6906,0.6298,0.6200,0.7770,0.5158,0.0130,non
6,logistic_regression,logistic_regression_baseline,0.500000,0.8773,0.8639,0.6912,0.6054,0.5997,0.5175,0.7129,0.0134,non
7,logistic_regression,logistic_regression_imbalance_default,0.500000,0.8773,0.8639,0.6912,0.6052,0.6001,0.5182,0.7129,0.0134,non


Le tableau est ordonné par AUC-ROC Test décroissante, puis par PR-AUC Test, puis par F1-score Test. 

Le recall et la précision restent affichés à titre diagnostique mais n'interviennent plus dans le tri.

### 5.1. Meilleure variante par famille

In [14]:
if RESULTATS:
    recap = (pd.DataFrame(RESULTATS.values())
             .sort_values("AUC-ROC Test", ascending=False)
             .reset_index(drop=True))

    # Critère de départage entre variantes à AUC-ROC Test égale : PR-AUC Test,
    # l'indicateur complémentaire retenu en 5.2 du rapport (le recall seul
    # n'est plus utilisé comme critère de décision, seulement diagnostique)
    meilleurs = (recap.sort_values(["AUC-ROC Test", "PR-AUC Test"],
                                    ascending=[False, False])
                      .groupby("Famille", as_index=False).first()
                      .sort_values("AUC-ROC Test", ascending=False))

    display(meilleurs[["Famille", "Variante", "Modèle", "AUC-ROC Test", "PR-AUC Test",
                       "F1 Test", "Recall Test", "Précision Test",
                       "Surapprentissage"]]
            .style.format({c: "{:.4f}" for c in
                           ["AUC-ROC Test", "PR-AUC Test", "F1 Test",
                            "Recall Test", "Précision Test"]})
            .background_gradient(subset=["AUC-ROC Test", "PR-AUC Test", "F1 Test"], cmap="Greens"))

,Famille,Variante,Modèle,AUC-ROC Test,PR-AUC Test,F1 Test,Recall Test,Précision Test,Surapprentissage
1,xgboost,EXP-007_02_frozen__chronological,XGBClassifier,0.8966,0.7619,0.6823,0.7124,0.6546,modéré
0,logistic_regression,logistic_regression_interactions,LogisticRegression,0.8692,0.6990,0.6404,0.6513,0.6298,non


Le tableau est ordonné par AUC-ROC Test décroissante, puis par PR-AUC Test, puis par F1-score Test. 

Le recall et la précision restent affichés à titre diagnostique mais n'interviennent plus dans le tri.

### 5.2. Export

Le tableau est exporté en CSV pour être repris directement dans le rapport.

In [15]:
if RESULTATS:
    EXPORT_PATH = Path("../reports/synthese_modeles.csv")
    EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    recap.to_csv(EXPORT_PATH, index=False)
    print(f"Exporté → {EXPORT_PATH}  ({len(recap)} lignes)")

Exporté → ..\reports\synthese_modeles.csv  (8 lignes)
